In [ ]:
!pip install -q groq gradio requests

### Data and Py basics

In [ ]:
# Task 1
tickets = [
    {"id": 1, "student": "Asha",   "message": "I cannot log in. It says my password is wrong even after reset."},
    {"id": 2, "student": "Ravi",   "message": "My payment was deducted twice but I got only one receipt."},
    {"id": 3, "student": "Meena",  "message": "I did not understand how temperature changes the LLM output."},
    {"id": 4, "student": "Kiran",  "message": "Great sessions so far, the trainer explains very clearly. Thank you!"},
    {"id": 5, "student": "Divya",  "message": "Login not working"},
    {"id": 6, "student": "Arjun",  "message": "Refund please"},
    {"id": 7, "student": "Sneha",  "message": "The Colab link for Day 1 notes shows access denied. Please share it again."},
    {"id": 8, "student": "Vikram", "message": "Please add more practice problems on JSON and REST APIs."},
]
print(len(tickets), "tickets created")

8 tickets created


In [ ]:
# Task 2
categories = {"login", "payment", "course doubt", "feedback", "other", "login", "payment"}
print(sorted(categories))

['course doubt', 'feedback', 'login', 'other', 'payment']


In [ ]:
# Task 3
for t in tickets:
  if len(t["message"].split()) < 4:
    t["flag"] = "need more detail"
  else:
    t["flag"] = "ok"
  print(t["id"], t["student"], "->", t["flag"])

1 Asha -> ok
2 Ravi -> ok
3 Meena -> ok
4 Kiran -> ok
5 Divya -> need more detail
6 Arjun -> need more detail
7 Sneha -> ok
8 Vikram -> ok


### JSON, files and REST API

In [ ]:
# Task 4
import json

with open("tickets.json", "w") as f:
  json.dump(tickets, f, indent=2)

with open("tickets.json", "r") as f:
  loaded = json.load(f)

print("Loaded", len(loaded), "tickets")

Loaded 8 tickets


In [10]:
# Task 5
import requests

all_tickets = list(tickets)

# try:
#   r = requests.get("https://jsonplaceholder.typicode.com/posts", timeout=10)
#   if r.status_code == 200:
#     all_tickets.append({
#         "id": 100 + post["id"],
#         "student": "API user",
#         "message": post["title"] + ". " + post["body"],
#         "flag": "ok",
#     })
#     print("Fetched 5 posts. Status: ", r.status_code)
#   else:
#     print("Request failed. Status code: ", r.status_code)
try:
    r = requests.get("https://jsonplaceholder.typicode.com/posts", timeout=10)
    if r.status_code == 200:
        for post in r.json()[:5]:
            all_tickets.append({
                "id": 100 + post["id"],
                "student": "API user",
                "message": post["title"] + ". " + post["body"],
                "flag": "ok",
            })
        print("Fetched 5 posts. Status:", r.status_code)
    else:
        print("Request failed. Status code:", r.status_code)
except Exception as e:
  print("could not reach the API: ", e)
print("Total tickets: ", len(all_tickets))

Fetched 5 posts. Status: 200
Total tickets:  13


### Keys and the LLM

In [12]:
# Task 6
import os

try:
  from google.colab import userdata
  api_key =  userdata.get("GROQ_API_KEY")
except Exception:
  api_key = os.environ.get("GROQ_API_KEY")

if api_key:
  print("API key found.")
else:
  print("GROQ_API_KEY missing.")

API key found.


In [21]:
# Task 7
from groq import Groq

# MODEL_NAME = "llama-3.3-70b-versatile"
MODEL_NAME = "qwen/qwen3.8-27b"

client = Groq(api_key=api_key) if api_key else None

def call_llm(prompt, temperature=0.2):
  response=client.chat.completions.create(
      model=MODEL_NAME,
      messages=[{"role": "user", "content": prompt}],
      temperature=temperature,
  )
  return response.choices[0].message.content, response.usage

def mock_llm(ticket_text):
  text = ticket_text.lower()
  if "log" in text or "password" in text:
    category, priority = "login", "high"
  elif "payment" in text or "refund" in text:
    category, priority = "payment", "hight"
  elif "understand" in text or "practice" in text:
    category, priority = "course doubt", "medium"
  elif "thank" in text or "great" in text:
    category, priority = "feedback", "low"
  else:
    category, priority = "other", "low"
  return json.dumps({
      "category": category,
      "priority": priority,
      "summary": "SIMULATED: looks like a " + category + " ticket.",
      "suggested_reply": "SIMULATED: Thanks for writing. We will help you soon.",
  })

In [23]:
# Task 8
def make_prompt(message, style):
  return (
        "Triage this student support ticket.\n"
        "Ticket: " + message + "\n\n"
        "Reply ONLY with JSON having these keys:\n"
        "category (one of: login, payment, course doubt, feedback, other),\n"
        "priority (low, medium or high),\n"
        "summary (one short sentence),\n"
        "suggested_reply (2 sentences, " + style + " tone)."
    )
def parse_json(text):
  text = text.replace("```json", "").replace("```", "").strip()
  try:
    return json.loads(text), None
  except json.JSONDecodeError:
    return None, "The model did not return valid JSON. Please try again."
msg = tickets[0]["message"]

if client:
  reply, usage = call_llm(make_prompt(msg, "friendly"), temperature=0.2)
else:
  reply, usage = mock_llm(msg), None

data, error = parse_json(reply)
print(error if error else data)

{'category': 'login', 'priority': 'high', 'summary': 'Student is unable to log in despite resetting their password.', 'suggested_reply': "I'm sorry to hear you're having trouble accessing your account. Please try clearing your browser cache or using a different device, and let us know if the issue persists so we can investigate further."}


In [24]:
# Task 9
if usage:
  print("Prompt tokens:", usage.prompt_tokens)
  print("Reply tokens:", usage.completion_tokens)
  print("Total tokens:", usage.total_tokens)
else:
  print("Mock mode: no real token usage.")

print("reply tokens: ", len(reply)//4)

Prompt tokens: 90
Reply tokens: 84
Total tokens: 174
reply tokens:  87


# Gradio

In [27]:
# Task 10: Gradio app
import gradio as gr

choices = [str(t["id"]) + " | " + t["student"] + " | " + t["message"][:50] for t in all_tickets]

def triage(choice, style, mock):
    ticket = all_tickets[choices.index(choice)]

    if not mock and client is None:
        return "No API key found. Tick Mock mode or add GROQ_API_KEY.", ""

    # Get the LLM (or mock) answer
    if mock:
        reply = mock_llm(ticket["message"])
    else:
        reply, _ = call_llm(make_prompt(ticket["message"], style), temperature=0.2)

    data, error = parse_json(reply)
    if error:
        return error, ""

    status = "Done (SIMULATED)" if mock else "Done (live: " + MODEL_NAME + ")"
    if ticket["flag"] == "needs more detail":
        status += " | Flag: needs more detail"

    result = (
        "### Ticket " + str(ticket["id"]) + " from " + ticket["student"] + "\n"
        "**Category:** " + data["category"] + "\n\n"
        "**Priority:** " + data["priority"] + "\n\n"
        "**Summary:** " + data["summary"] + "\n\n"
        "**Suggested reply:** " + data["suggested_reply"]
    )
    return status, result

demo = gr.Interface(
    fn=triage,
    inputs=[
        gr.Dropdown(choices, value=choices[0], label="Ticket"),
        gr.Dropdown(["friendly", "formal", "concise"], value="friendly", label="Reply style"),
        gr.Checkbox(value=(client is None), label="Mock mode"),
    ],
    outputs=[gr.Textbox(label="Status"), gr.Markdown(label="Result")],
    title="TicketTriage",
    submit_btn="Triage",
)
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8bf23e7ceb873db63b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
